# 01. Xác minh NF-ToN-IoT-v3 (task 1.2)

Chạy trên Kaggle, gắn dataset `athena21/netflow-v3-datasets`. Link tải chính thức của UQ (`rdm.uq.edu.au`) yêu cầu đăng nhập (API trả 401), nên dùng bản đăng lại này và **đối chiếu số dòng, số mẫu theo lớp với số liệu UQ công bố** để chứng minh là bản gốc.

- Trang chính thức: https://staff.itee.uq.edu.au/marius/NIDS_datasets/
- Bản ghi dữ liệu: Luay M. và cộng sự (2025). *NF-ToN-IoT-v3*. The University of Queensland. https://doi.org/10.48610/44d7c5e
- Bài báo: Luay M., Layeghy S., Hosseininoorbin S., Sarhan M., Moustafa N., Portmann M. *Temporal Analysis of NetFlow Datasets for Network Intrusion Detection Systems*. arXiv:2503.04404 (2025).

Số liệu UQ công bố cho NF-ToN-IoT-v3: 27.520.260 flow; 16.792.214 benign (61,02%); 10.728.046 tấn công (38,98%); 9 loại tấn công.

Tiêu chí 2–3 của task 1.2: có cột timestamp (in 5 dòng làm bằng chứng); tỉ lệ dòng có ts tăng dần và khoảng thời gian bao phủ.

In [ ]:
import glob, hashlib, json, os, time
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.csv as pacsv
import pyarrow.compute as pc

OFFICIAL = {"rows": 27_520_260, "benign": 16_792_214, "attack": 10_728_046}
OUT = "/kaggle/working/nf_ton_iot_v3_check.json"
summary = {"source": "kaggle:athena21/netflow-v3-datasets", "official": OFFICIAL}

files = sorted(glob.glob("/kaggle/input/**/*.csv", recursive=True))
for f in files:
    print(f"{os.path.getsize(f) / 2**30:7.2f} GiB  {f}")
CSV = next(f for f in files if os.path.basename(f) == "NF-ToN-IoT-v3.csv")
FEAT = next((f for f in files if os.path.basename(f) == "NetFlow_v3_Features.csv"), None)
summary["file"] = {"path": CSV, "bytes": os.path.getsize(CSV)}

## Cột và 5 dòng đầu (bằng chứng tiêu chí 2)

In [ ]:
head = pd.read_csv(CSV, nrows=5)
cols = list(head.columns)
print(f"{len(cols)} cột:", cols)
summary["columns"] = cols
summary["n_columns"] = len(cols)
TIME_COLS = ["FLOW_START_MILLISECONDS", "FLOW_END_MILLISECONDS"]
print("Có cột thời gian:", {c: c in cols for c in TIME_COLS})
with pd.option_context("display.max_columns", None, "display.width", 250):
    print(head.T)
summary["head5"] = head.astype(str).to_dict(orient="records")
if FEAT:
    feat = pd.read_csv(FEAT)
    print(feat.head(60).to_string())

## Đọc các cột cần cho kiểm tra

In [ ]:
LABEL_COLS = [c for c in ("Label", "Attack") if c in cols]
NEED = TIME_COLS + ["IPV4_SRC_ADDR", "IPV4_DST_ADDR"] + LABEL_COLS
t0 = time.time()
tbl = pacsv.read_csv(
    CSV,
    read_options=pacsv.ReadOptions(block_size=64 << 20),
    convert_options=pacsv.ConvertOptions(
        include_columns=NEED,
        column_types={"FLOW_START_MILLISECONDS": pa.int64(), "FLOW_END_MILLISECONDS": pa.int64()},
    ),
)
print(f"Đọc {tbl.num_rows:,} dòng x {tbl.num_columns} cột trong {time.time() - t0:.0f} s, "
      f"{tbl.nbytes / 2**30:.2f} GiB RAM")

## Đối chiếu số lượng với số liệu chính thức

In [ ]:
n = tbl.num_rows
label = tbl["Label"].to_numpy() if "Label" in LABEL_COLS else None
counts = {"rows": n}
if label is not None:
    counts["benign"] = int((label == 0).sum())
    counts["attack"] = int((label == 1).sum())
match = {k: counts.get(k) == v for k, v in OFFICIAL.items()}
print("Đếm được:", counts)
print("Khớp chính thức:", match)
summary["counts"] = counts
summary["match_official"] = match
if "Attack" in LABEL_COLS:
    per_class = pd.Series(tbl["Attack"].to_pandas()).value_counts()
    print(per_class.to_string())
    summary["per_class"] = {k: int(v) for k, v in per_class.items()}
summary["n_src_ip"] = int(pc.count_distinct(tbl["IPV4_SRC_ADDR"]).as_py())
summary["n_dst_ip"] = int(pc.count_distinct(tbl["IPV4_DST_ADDR"]).as_py())
print("IP nguồn khác nhau:", summary["n_src_ip"], "| IP đích khác nhau:", summary["n_dst_ip"])

## Kiểm tra thứ tự thời gian (tiêu chí 3)

In [ ]:
start = tbl["FLOW_START_MILLISECONDS"].to_numpy()
end = tbl["FLOW_END_MILLISECONDS"].to_numpy()
d = np.diff(start)
ts = {
    "null_start": int(tbl["FLOW_START_MILLISECONDS"].null_count),
    "min_start_utc": str(pd.to_datetime(start.min(), unit="ms", utc=True)),
    "max_start_utc": str(pd.to_datetime(start.max(), unit="ms", utc=True)),
    "span_hours": float((start.max() - start.min()) / 3.6e6),
    "frac_nondecreasing": float((d >= 0).mean()),
    "frac_strictly_increasing": float((d > 0).mean()),
    "max_backward_jump_s": float(-d.min() / 1000) if (d < 0).any() else 0.0,
    "frac_end_ge_start": float((end >= start).mean()),
    "duration_ms_p50": float(np.percentile(end - start, 50)),
    "duration_ms_p99": float(np.percentile(end - start, 99)),
}
for k, v in ts.items():
    print(f"{k:26s}: {v}")
summary["timestamps"] = ts

## Phân bố theo ngày (đối chiếu Bảng 4 của bài báo: 10 ngày, 3 ngày đầu chỉ benign)

In [ ]:
day = pd.to_datetime(start, unit="ms", utc=True).floor("D")
frame = pd.DataFrame({"day": day, "attack": tbl["Attack"].to_pandas() if "Attack" in LABEL_COLS else label})
by_day = frame.groupby(["day", "attack"]).size().unstack(fill_value=0)
with pd.option_context("display.max_columns", None, "display.width", 250):
    print(by_day)
summary["by_day"] = {str(k.date()): {c: int(v) for c, v in row.items() if v} for k, row in by_day.iterrows()}
del frame, day

## SHA-256 của file CSV (để truy vết)

In [ ]:
t0 = time.time()
h = hashlib.sha256()
with open(CSV, "rb") as f:
    for chunk in iter(lambda: f.read(64 << 20), b""):
        h.update(chunk)
summary["file"]["sha256"] = h.hexdigest()
print(summary["file"], f"({time.time() - t0:.0f} s)")

In [ ]:
with open(OUT, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=1, default=str)
print("Đã ghi", OUT)
print(json.dumps({k: summary[k] for k in ("counts", "match_official", "timestamps")}, indent=1, default=str))